# 06 — LoRA (option A) : des légendes de VLM pour entraîner une LoRA
**Question :** des légendes écrites par un petit VLM, même quantifié en 4 bits, permettent-elles d'entraîner une LoRA de Stable Diffusion aussi bien que des légendes humaines ?

**Étape 7 — Préparer les données.** On garde les photos de chiens de nos 500 images, puis on crée 3 jeux identiques en images mais différents en légendes : `humain`, `vlm2_normal`, `vlm2_4bit`.

**GPU T4 obligatoire à partir de l'étape 8.** Dans une nouvelle session, relancez toutes les cellules depuis le début : l'étape 7 se refait en quelques secondes.

In [ ]:
# Installation (Google Colab / Kaggle avec GPU)
!git clone https://github.com/doaemalak/VLMs-l-gers-pour-l-gender-ou-juger-des-images.git
%cd VLMs-l-gers-pour-l-gender-ou-juger-des-images
!pip install -q -r requirements.txt

# Colab efface tout entre deux sessions : on retélécharge les 500 mêmes images (seed 42)
![ -f data/subset/subset.csv ] || python data/download.py --config configs/data.yaml

# Colab préinstalle une vieille version de torchao, refusée par peft (outil LoRA).
# Nous n'utilisons pas torchao : on le retire.
!pip uninstall -y -q torchao

In [ ]:
# Les résultats sont enregistrés dans Google Drive (Mon Drive/projet7_results)
# pour ne pas être perdus si Colab coupe la session.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/projet7_results
!rm -rf results && ln -s /content/drive/MyDrive/projet7_results results
!ls results

In [ ]:
from src.lora_data import prepare
recap = prepare('configs/lora.yaml')
recap.head(10)

## Exemples : la même image avec ses 3 légendes d'entraînement

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src.utils import load_image
subset = pd.read_csv('data/subset/subset.csv').set_index('image')
for image, r in recap.head(3).iterrows():
    plt.imshow(load_image(subset.loc[image, 'path'])); plt.axis('off'); plt.show()
    for source in ['humain', 'vlm2_normal', 'vlm2_4bit']:
        print(f'{source:12s}:', r[source])

## Phrases de test (pour l'étape 9)
Légendes humaines d'images de chiens **jamais vues** pendant l'entraînement.

In [ ]:
pd.read_csv('results/lora/prompts_test.csv')

# Étape 8 — Entraîner les 3 LoRA
Mêmes images, mêmes réglages, même graine : **seule la légende change**.

Les LoRA vont dans Google Drive (`projet7_results/lora/`). Si Colab coupe, relancez tout : les LoRA déjà terminées sont sautées.

### Test rapide (environ 2 minutes)
Vérifie que tout fonctionne avant de lancer les vrais entraînements.

In [ ]:
from src.train_lora import train
train('configs/lora.yaml', 'humain', max_steps=5)

### Les 3 vrais entraînements (environ 20 à 30 minutes chacun)

In [ ]:
for source in ['humain', 'vlm2_normal', 'vlm2_4bit']:
    train('configs/lora.yaml', source)

### Courbes d'apprentissage et tableau
La *loss* (erreur) doit globalement baisser, même si elle reste très irrégulière : c'est normal en diffusion, car le niveau de bruit est tiré au hasard à chaque étape.

In [ ]:
import json
fig, ax = plt.subplots(figsize=(8, 4))
rows = []
for source in ['humain', 'vlm2_normal', 'vlm2_4bit']:
    loss = pd.read_csv(f'results/lora/{source}/loss.csv')
    ax.plot(loss['step'], loss['loss'].rolling(20).mean(), label=source)  # moyenne glissante
    rows.append(json.load(open(f'results/lora/{source}/resume.json')))
ax.set_xlabel('mise à jour'); ax.set_ylabel('loss (moyenne sur 20)'); ax.legend()
plt.savefig('results/lora/courbes_loss.png', bbox_inches='tight'); plt.show()
pd.DataFrame(rows)